# 03 — Condition dependence (Layers 2–4) and recording-batch effect

Input: the outputs of `02_features_group_comparison.ipynb` (`features_<COND>.csv`,
`layer1_summary.csv`) and the file inventory of `00_duplicate_audit.ipynb`
(`file_inventory.csv`, for the recording dates). No EEG feature is recomputed here. Layers 2
and 3 are in `src/condition_tests.py`, shared with `04_stability_sensitivity.ipynb`.

| Step | Question | Method |
|---|---|---|
| Layer 2 | Does the HC–MDD difference change between EC, EO and TASK? | Group × Condition interaction: closed-form F, p by permutation of group labels between subjects (main) and from F(2, 2(N−2)); FDR over features. Post hoc contrasts from a linear mixed model (REML), t with 2(N−2) df, Holm within feature |
| Layer 3 | Is the HC–MDD difference equivalent across conditions? | TOST on the three contrasts with margin Δ = ±0.30 SD; global p = largest of the three; FDR over features. Sensitivity to Δ = 0.20 and 0.50, smallest attainable margin, minimum detectable effect |
| Layer 4 | Do domains differ? | Observed proportions of formal status by domain; participant-level stratified bootstrap of the median group effect and of its range across conditions; feature redundancy (correlation clusters, effective dimension) |
| Batch | Do the group differences follow the recording session of the controls? | Cliff's delta of MDD vs all HC, MDD vs HC recorded outside the January 2013 session, and January vs other HC |

**Outputs (in `RESULTS_DIR`):** `layer2_interaction.csv`, `layer3_equivalence.csv`,
`layer3_margin_by_feature.csv`, `layer3_margin_summary.csv`,
`layer3_margin_by_domain.csv`, `descriptive_vs_formal.csv`, `layer4_*.csv`,
`batch_effect_by_feature.csv`, `batch_effect_by_family.csv`.

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
import sys
if 'google.colab' in sys.modules:
    !pip install -q statsmodels
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '..'   # repository root (on Colab: the folder where the repository was cloned or copied)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
import time
import warnings

import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.stats import rankdata, spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

import condition_tests as C
from condition_tests import ALPHA, CONTRASTS, EQUIV_MARGIN, tost_p
from eeg_features import CONDITIONS, cliffs_delta, domain_of, family_of

warnings.filterwarnings('ignore')
print(f'Python {sys.version.split()[0]}')
for module in (np, scipy, pd, statsmodels, sklearn):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
RESULTS_DIR = 'path/to/results'     # outputs of 02_features_group_comparison.ipynb
AUDIT_DIR = 'path/to/audit_outputs' # outputs of 00_duplicate_audit.ipynb (file_inventory.csv)

print('Layers 2-3 (src/condition_tests.py):', C.parameters())

# --- Layer 3 ------------------------------------------------------------------
SENSITIVITY_MARGINS = [0.20, 0.30, 0.50]
MARGIN_GRID = np.round(np.arange(0.01, 3.0001, 0.01), 2)   # for the smallest margin with FDR
POWER = 0.80                           # for the minimum detectable effect

# --- Layer 4 ------------------------------------------------------------------
BOOTSTRAP_N = 2000
BOOTSTRAP_SEED = 20260930
CORR_CLUSTER_THRESHOLD = 0.80          # features with |rho| >= 0.80 are clustered
PCA_VARIANCE_TARGET = 0.90

# --- Batch effect -------------------------------------------------------------
BATCH_SESSION = ('2013-01-01', '2013-01-31')   # recording session of most controls

## Data

In [ ]:
long_data = C.load_long_features(RESULTS_DIR)
layer1 = pd.read_csv(os.path.join(RESULTS_DIR, 'layer1_summary.csv'), float_precision='round_trip')
FEATURES = [c for c in long_data.columns if c not in ('subject', 'group', 'condition')]
n_hc = long_data.loc[long_data.group == 'HC', 'subject'].nunique()
n_mdd = long_data.loc[long_data.group == 'MDD', 'subject'].nunique()
print(f'{n_hc} HC + {n_mdd} MDD, {len(FEATURES)} features, {len(long_data)} rows')

## Layer 2 — Group × Condition interaction

Each feature is log10-transformed when its prefix is in `LOG_PREFIXES` and all values are
positive, then z-scored over all subjects and conditions.

**Main test.** With $P_{ij} = y_{ij} - \bar{y}_{i\cdot}$ the subject-centered profile,
the interaction F is computed in closed form. In a complete design with a random
intercept it equals the joint Wald statistic of the two interaction terms of the REML mixed
model divided by 2, and is referred to $F(K-1, (N-2)(K-1))$. The main p-value is obtained by
permuting the HC/MDD labels between subjects (the three conditions of a subject move
together), with the same `PERM_N` permutations for all features:
$p = (1 + \#\{F_b \ge F_{obs}\}) / (B + 1)$.

**Post hoc contrasts.** From `feature ~ group * condition + (1 | subject)` (REML, HC and
EC as references): EO−EC, TASK−EC and TASK−EO changes of the MDD−HC difference, with t
tests on $2(N-2)$ df and Holm correction within each feature.

In [ ]:
layer2 = C.run_layer2(long_data, FEATURES, save_dir=RESULTS_DIR)

p, pF = layer2['p_interaction_perm'], layer2['p_interaction_F']
q_F = multipletests(pF.dropna(), method='fdr_bh')[1]
print(f"Converged mixed models: {int(layer2.model_ok.sum())}/{len(layer2)} | "
      f"log10: {int(layer2.transform_perm.eq('log10').sum())} | "
      f"random-intercept variance at 0: {int(layer2.random_intercept_boundary.sum())}")
print(f'Significant interactions (FDR, permutation): {int(layer2.interaction_significant.sum())} | '
      f'with the F p-value: {int((q_F < ALPHA).sum())}')
print(f'Smallest permutation p: {p.min():.4f} ({layer2.loc[p.idxmin(), "feature"]}) | '
      f'smallest q: {layer2.p_interaction_fdr.min():.3f}')
print(f'Nominal p < 0.05: {int((p < 0.05).sum())} (expected under the global null: {0.05 * p.notna().sum():.0f}) | '
      f'p < 0.01: {int((p < 0.01).sum())} (expected: {0.01 * p.notna().sum():.0f})')
print(f'Permutation vs F p-values: Pearson r = {np.corrcoef(p, pF)[0, 1]:.3f} | '
      f'F: {int((pF < 0.05).sum())} below 0.05, smallest {pF.min():.4f}')

## Layer 3 — Equivalence across conditions

TOST on each contrast with margin $\pm\Delta$ (t with $2(N-2)$ df): equivalent when the 90%
CI lies within $[-\Delta, +\Delta]$. Global equivalence requires all three contrasts; the
global p-value is the largest of the three TOST p-values, corrected by FDR over features.
Formal status: condition-dependent (interaction, no equivalence), condition-equivalent
(equivalence, no interaction), inconclusive (neither).

In [ ]:
layer3 = C.run_layer3(layer2, save_dir=RESULTS_DIR)
print(layer3['formal_condition_status'].value_counts().to_string())
print(f"Smallest q (equivalence, margin {EQUIV_MARGIN}): {layer3.p_equivalence_fdr.min():.3f}")
print('Equivalent without correction, per contrast:',
      {c: int(layer3[f'equiv_raw_{c}'].sum()) for c in CONTRASTS})

### Precision, sensitivity to the margin and minimum detectable effect

- Precision: median SE and median half-width of the 90% CI of the contrasts.
- Minimum detectable effect (80% power, two-sided α = 0.05) of a single contrast:
  $\mathrm{MDE} = (t_{0.975;\nu} + t_{0.80;\nu})\,\mathrm{SE}$, $\nu = 2(N-2)$.
- Number of equivalent features with Δ = 0.20, 0.30 and 0.50 SD.
- Smallest attainable margin per feature: without correction, the smallest Δ whose
  interval contains the 90% CI of the three contrasts; with FDR, the smallest Δ on
  `MARGIN_GRID` with q < 0.05.

In [ ]:
EST = layer3[[f'delta_effect_{c}_z' for c in CONTRASTS]].to_numpy(float)
SE = layer3[[f'se_{c}_z' for c in CONTRASTS]].to_numpy(float)
NU = layer3['df_resid'].to_numpy(float)
VALID = np.isfinite(EST).all(1) & np.isfinite(SE).all(1) & (SE > 0).all(1) & (NU > 0)


def equivalence_at(margin):
    p_global = tost_p(EST, SE, NU[:, None], margin).max(axis=1)
    q = np.full(len(layer3), np.nan)
    q[VALID] = multipletests(p_global[VALID], method='fdr_bh')[1]
    p_global[~VALID] = np.nan
    return p_global, q


# Check: same global p as Layer 3 at the main margin.
assert np.nanmax(np.abs(equivalence_at(EQUIV_MARGIN)[0] - layer3['p_equivalence_global'])) < 1e-9

print('Precision of the contrasts (SD units):')
t90 = stats.t.ppf(1 - ALPHA, NU[VALID])
for k, c in enumerate(CONTRASTS):
    half = t90 * SE[VALID, k]
    print(f'  {c:11s} median SE {np.median(SE[VALID, k]):.3f} | median 90% CI half-width {np.median(half):.3f} '
          f'(P10-P90 {np.percentile(half, 10):.3f}-{np.percentile(half, 90):.3f})')

mde = (stats.t.ppf(0.975, NU[VALID])[:, None] + stats.t.ppf(POWER, NU[VALID])[:, None]) * SE[VALID]
nu_med = np.median(NU[VALID])
print(f'Minimum detectable effect ({POWER:.0%} power): {(stats.t.ppf(0.975, nu_med) + stats.t.ppf(POWER, nu_med)) * np.median(SE[VALID]):.2f} SD '
      f'at the median SE (P10-P90 over features and contrasts: {np.percentile(mde, 10):.2f}-{np.percentile(mde, 90):.2f}) | '
      f'median |estimate| per contrast: {np.round(np.nanmedian(np.abs(EST[VALID]), axis=0), 2)}')

delta_min_raw = np.where(VALID, (np.abs(EST) + stats.t.ppf(1 - ALPHA, NU)[:, None] * SE).max(axis=1), np.nan)
delta_min_fdr = np.full(len(layer3), np.nan)
for m in MARGIN_GRID:
    new = np.isnan(delta_min_fdr) & (equivalence_at(m)[1] < ALPHA)
    delta_min_fdr[new] = m
    if not np.isnan(delta_min_fdr[VALID]).any():
        break

margin_by_feature = pd.DataFrame({'feature': layer3.feature, 'domain': layer3.domain,
                                  'delta_min_raw': delta_min_raw, 'delta_min_fdr': delta_min_fdr})
summary_rows, domain_rows = [], []
for m in SENSITIVITY_MARGINS:
    pg, q = equivalence_at(m)
    tag = f'{m:.2f}'.replace('.', '')
    margin_by_feature[f'p_global_{tag}'], margin_by_feature[f'q_global_{tag}'] = pg, q
    margin_by_feature[f'equiv_raw_{tag}'], margin_by_feature[f'equiv_fdr_{tag}'] = pg < ALPHA, q < ALPHA
    summary_rows.append({'margin_sd': m, 'equivalent_uncorrected': int((pg < ALPHA).sum()),
                         'equivalent_fdr': int((q < ALPHA).sum()), 'smallest_q': np.nanmin(q)})
    g = margin_by_feature[VALID].groupby('domain')
    domain_rows.append(pd.DataFrame({'margin_sd': m, 'n_testable': g.size(),
                                     'equivalent_uncorrected': g[f'equiv_raw_{tag}'].sum(),
                                     'equivalent_fdr': g[f'equiv_fdr_{tag}'].sum()}))
margin_summary = pd.DataFrame(summary_rows)
margin_by_domain = pd.concat(domain_rows).reset_index()
margin_by_feature.to_csv(os.path.join(RESULTS_DIR, 'layer3_margin_by_feature.csv'), index=False)
margin_summary.to_csv(os.path.join(RESULTS_DIR, 'layer3_margin_summary.csv'), index=False)
margin_by_domain.to_csv(os.path.join(RESULTS_DIR, 'layer3_margin_by_domain.csv'), index=False)

print('\nEquivalence across the three conditions by margin:')
print(margin_summary.round(3).to_string(index=False))
for name, v in (('uncorrected', delta_min_raw[VALID]), ('FDR', delta_min_fdr[VALID])):
    v = v[np.isfinite(v)]
    print(f'Smallest attainable margin ({name}): min {v.min():.2f} | median {np.median(v):.2f} SD')
print('Median smallest margin by domain:')
print(margin_by_feature[VALID].groupby('domain')[['delta_min_raw', 'delta_min_fdr']].median().round(2).to_string())
print('Features with the smallest margins:')
print(margin_by_feature[VALID].nsmallest(10, 'delta_min_raw')[['feature', 'delta_min_raw', 'delta_min_fdr']]
      .round(3).to_string(index=False))

### Descriptive detection pattern (Layer 1) versus formal tests (Layers 2–3)

In [ ]:
merged = layer1.merge(layer3[['feature', 'p_interaction_perm', 'interaction_significant',
                              'equivalence_significant', 'formal_condition_status']], on='feature')
labels = {3: 'all three conditions', 2: 'two conditions', 1: 'one condition only', 0: 'none'}
merged['pattern'] = merged.n_conditions_detected.map(labels)
merged['nominal'] = merged.p_interaction_perm < 0.05
agg = dict(n=('feature', 'size'), interaction_fdr=('interaction_significant', 'sum'),
           interaction_nominal=('nominal', 'sum'), equivalent_fdr=('equivalence_significant', 'sum'))
desc_formal = merged.groupby('pattern').agg(**agg).reindex(list(labels.values())).fillna(0).astype(int)
desc_formal.loc['total'] = desc_formal.sum()
desc_formal['interaction_nominal_pct'] = (100 * desc_formal.interaction_nominal / desc_formal.n).round(1)
desc_formal.to_csv(os.path.join(RESULTS_DIR, 'descriptive_vs_formal.csv'))
print(desc_formal.to_string())

## Layer 4 — Robustness by domain

- **Observed proportions** of formal status by domain, over all testable features and over
  the features detected in at least one condition (no CI: bootstrapping counts of
  significant tests inflates power through duplicated subjects).
- **Continuous quantities by domain**, from the Cliff's delta of each feature in the three
  conditions: the group effect $|\bar\delta|$ (mean of the three deltas, in absolute value)
  and its range across conditions ($\max - \min$), summarized by the median over features.
  Participant-level bootstrap stratified by group (the three conditions of a subject stay
  together), basic 95% CI ($2\hat\theta - q_{0.975}$, $2\hat\theta - q_{0.025}$), and
  differences between domains (descriptive).
- **Redundancy:** hierarchical clustering of features within each domain (Spearman over
  the subject means of the three conditions, average linkage, distance $1 - |\rho|$,
  threshold $|\rho| \ge 0.80$) and effective dimension (principal components for 90% of
  the variance).

In [ ]:
def domain_proportions(layer3, layer1):
    d = layer3.merge(layer1[['feature', 'n_conditions_detected']], on='feature', how='left')
    d['detected'] = d['n_conditions_detected'].fillna(0) > 0
    d = d[d['formal_condition_status'].ne('not-tested')]
    rows = []
    for dom, g in d.groupby('domain'):
        gd = g[g.detected]
        row = {'domain': dom, 'n_testable': len(g), 'n_detected': len(gd)}
        for status in ('condition-dependent', 'condition-equivalent', 'inconclusive'):
            key = status.replace('condition-', '')
            row[f'n_{key}'] = int(g.formal_condition_status.eq(status).sum())
            row[f'prop_{key}_all'] = g.formal_condition_status.eq(status).mean()
            row[f'prop_{key}_detected'] = gd.formal_condition_status.eq(status).mean() if len(gd) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def subject_means(data):
    # Mean of EC, EO and TASK per subject, so that each subject counts once.
    return data.groupby('subject', observed=True)[FEATURES].mean()


def correlation_clusters(data, threshold=CORR_CLUSTER_THRESHOLD):
    X = subject_means(data)
    rows = []
    for dom in sorted({domain_of(f) for f in FEATURES}):
        fd = [f for f in FEATURES if domain_of(f) == dom]
        good = [f for f in fd if X[f].notna().sum() >= 4 and X[f].nunique() > 1]
        if len(good) > 1:
            C = X[good].corr(method='spearman', min_periods=4).abs().fillna(0.0).clip(0, 1).to_numpy(copy=True)
            np.fill_diagonal(C, 1.0)
            labels = fcluster(linkage(squareform(np.clip(1.0 - C, 0, 1), checks=False), method='average'),
                              t=1.0 - threshold, criterion='distance')
        else:
            labels = np.ones(len(good), dtype=int)
        sizes = pd.Series(labels).value_counts()
        rows += [{'domain': dom, 'feature': f, 'corr_cluster': int(l), 'cluster_size': int(sizes[l]),
                  'family': family_of(f)} for f, l in zip(good, labels)]
        rows += [{'domain': dom, 'feature': f, 'corr_cluster': np.nan, 'cluster_size': np.nan,
                  'family': family_of(f)} for f in fd if f not in good]
    return pd.DataFrame(rows)


def effective_dimension(data, target=PCA_VARIANCE_TARGET):
    X = subject_means(data)
    rows = []
    for dom in sorted({domain_of(f) for f in FEATURES}):
        fd = [f for f in FEATURES if domain_of(f) == dom]
        good = [f for f in fd if X[f].notna().sum() >= 4 and X[f].nunique() > 1]
        Z = StandardScaler().fit_transform(X[good].fillna(X[good].median()))
        n_comp = int(np.searchsorted(np.cumsum(PCA().fit(Z).explained_variance_ratio_), target) + 1)
        rows.append({'domain': dom, 'n_features': len(fd), 'n_features_used': len(good),
                     'n_components': n_comp, 'effective_fraction': n_comp / len(good)})
    return pd.DataFrame(rows)


def cliffs_delta_columns(hc, md):
    # Cliff's delta per column (positive = MDD > HC), from the Mann-Whitney U.
    nh, nm = len(hc), len(md)
    out = np.full(hc.shape[1], np.nan)
    ok = np.isfinite(hc).all(axis=0) & np.isfinite(md).all(axis=0)
    if ok.any():
        r = rankdata(np.vstack([hc[:, ok], md[:, ok]]), axis=0)
        out[ok] = 2.0 * (r[nh:].sum(axis=0) - nm * (nm + 1) / 2.0) / (nh * nm) - 1.0
    for j in np.where(~ok)[0]:
        a, b = hc[:, j][np.isfinite(hc[:, j])], md[:, j][np.isfinite(md[:, j])]
        if len(a) >= 3 and len(b) >= 3:
            out[j] = cliffs_delta(a, b)
    return out


def domain_metrics(X, is_mdd, idx, domains, domain_list):
    m = is_mdd[idx]
    D = np.vstack([cliffs_delta_columns(X[c][idx][~m], X[c][idx][m]) for c in CONDITIONS])
    effect = np.abs(np.nanmean(D, axis=0))
    rng_ = np.nanmax(D, axis=0) - np.nanmin(D, axis=0)
    out = {}
    for dom in domain_list:
        sel = domains == dom
        out[(dom, 'group_effect')] = float(np.nanmedian(effect[sel]))
        out[(dom, 'between_condition_range')] = float(np.nanmedian(rng_[sel]))
    return out


def bootstrap_domains(data, n_boot=BOOTSTRAP_N, seed=BOOTSTRAP_SEED):
    d = data.copy()
    d['condition'] = d['condition'].astype(str)
    subs = sorted(d.subject.unique())
    is_mdd = d.drop_duplicates('subject').set_index('subject').loc[subs, 'group'].astype(str).eq('MDD').to_numpy()
    X = {c: d[d.condition == c].set_index('subject').reindex(subs)[FEATURES]
               .apply(pd.to_numeric, errors='coerce').to_numpy(float) for c in CONDITIONS}
    domains = np.array([domain_of(f) for f in FEATURES])
    domain_list = sorted(set(domains))
    idx_hc, idx_md = np.where(~is_mdd)[0], np.where(is_mdd)[0]

    obs = domain_metrics(X, is_mdd, np.arange(len(subs)), domains, domain_list)
    rng = np.random.default_rng(seed)
    reps = [domain_metrics(X, is_mdd, np.concatenate([rng.choice(idx_hc, len(idx_hc)),
                                                      rng.choice(idx_md, len(idx_md))]), domains, domain_list)
            for _ in range(n_boot)]
    keys = list(obs)
    M = np.array([[r[k] for k in keys] for r in reps])

    rows = []
    for j, (dom, metric) in enumerate(keys):
        x = M[:, j][np.isfinite(M[:, j])]
        q_lo, q_hi = np.quantile(x, [0.025, 0.975])
        o = obs[(dom, metric)]
        rows.append({'domain': dom, 'metric': metric, 'observed': o, 'bootstrap_median': float(np.median(x)),
                     'bootstrap_bias': float(np.mean(x) - o),
                     'ci95_percentile_low': q_lo, 'ci95_percentile_high': q_hi,
                     'ci95_basic_low': 2 * o - q_hi, 'ci95_basic_high': 2 * o - q_lo,
                     'bootstrap_sd': float(np.std(x, ddof=1)), 'n_boot_valid': len(x)})
    diffs = []
    for metric in ('group_effect', 'between_condition_range'):
        for a in range(len(domain_list)):
            for b in range(a + 1, len(domain_list)):
                da, db = domain_list[a], domain_list[b]
                x = M[:, keys.index((da, metric))] - M[:, keys.index((db, metric))]
                x = x[np.isfinite(x)]
                o = obs[(da, metric)] - obs[(db, metric)]
                q_lo, q_hi = np.quantile(x, [0.025, 0.975])
                diffs.append({'metric': metric, 'domain_a': da, 'domain_b': db, 'diff_observed': o,
                              'ci95_basic_low': 2 * o - q_hi, 'ci95_basic_high': 2 * o - q_lo,
                              'ci_excludes_zero': bool(2 * o - q_hi > 0 or 2 * o - q_lo < 0)})
    raw = pd.DataFrame(M, columns=[f'{d}|{m}' for d, m in keys])
    return raw, pd.DataFrame(rows), pd.DataFrame(diffs)

In [ ]:
proportions = domain_proportions(layer3, layer1)
clusters = correlation_clusters(long_data)
eff_dim = effective_dimension(long_data)
t0 = time.time()
boot_raw, boot_summary, boot_diffs = bootstrap_domains(long_data)
print(f'Bootstrap: {BOOTSTRAP_N} resamples in {(time.time() - t0) / 60:.1f} min')

detected = set(layer1.loc[layer1.n_conditions_detected > 0, 'feature'])
cl = clusters.dropna(subset=['corr_cluster'])
redundancy = pd.DataFrame({
    'n_clusters': cl.groupby('domain')['corr_cluster'].nunique(),
    'n_clusters_with_detected': cl[cl.feature.isin(detected)].groupby('domain')['corr_cluster'].nunique(),
}).fillna(0).astype(int)

continuous = boot_summary.pivot(index='domain', columns='metric',
                                values=['observed', 'ci95_basic_low', 'ci95_basic_high'])
continuous.columns = [f'{m}_{v}' for v, m in continuous.columns]
domain_table = (proportions.set_index('domain').join(continuous).join(redundancy)
                .join(eff_dim.set_index('domain')[['n_components']]).reset_index())

proportions.to_csv(os.path.join(RESULTS_DIR, 'layer4_domain_proportions.csv'), index=False)
clusters.to_csv(os.path.join(RESULTS_DIR, 'layer4_feature_clusters.csv'), index=False)
eff_dim.to_csv(os.path.join(RESULTS_DIR, 'layer4_effective_dimension.csv'), index=False)
boot_raw.to_csv(os.path.join(RESULTS_DIR, 'layer4_bootstrap_raw.csv'), index=False)
boot_summary.to_csv(os.path.join(RESULTS_DIR, 'layer4_bootstrap_summary.csv'), index=False)
boot_diffs.to_csv(os.path.join(RESULTS_DIR, 'layer4_bootstrap_domain_differences.csv'), index=False)
domain_table.to_csv(os.path.join(RESULTS_DIR, 'layer4_domain_table.csv'), index=False)

cols = ['domain', 'n_testable', 'n_detected', 'prop_inconclusive_all', 'group_effect_observed',
        'group_effect_ci95_basic_low', 'group_effect_ci95_basic_high', 'between_condition_range_observed',
        'n_clusters', 'n_clusters_with_detected', 'n_components']
print(domain_table[cols].sort_values('group_effect_observed', ascending=False).round(2).to_string(index=False))
print(f'\nTotal: {int(redundancy.n_clusters.sum())} clusters ({int(redundancy.n_clusters_with_detected.sum())} '
      f'with detected features); summed effective dimension {int(eff_dim.n_components.sum())}')
print('\nDifferences between domains whose basic 95% CI excludes zero (descriptive):')
print(boot_diffs[boot_diffs.ci_excludes_zero].round(3).to_string(index=False))

## Recording-batch effect

The controls were recorded mostly in one session (January 2013) and the patients in
another period. The batch of each control is read from the recording dates in the EDF
headers (`file_inventory.csv`). For each feature significant in a condition, three Cliff's
deltas are computed: MDD versus all HC; MDD versus the HC recorded outside the January
session; and January HC versus the other HC (batch effect within the healthy group). Batch
and group are almost collinear, so the batch effect cannot be removed by adjustment; the
analysis identifies differences that cannot be attributed to the diagnosis.

In [ ]:
inventory = pd.read_csv(os.path.join(AUDIT_DIR, 'file_inventory.csv'), parse_dates=['start'])
hc_sample = sorted(long_data.loc[long_data.group == 'HC', 'subject'].unique())
dates = (inventory[inventory.subject_id.isin(hc_sample)]
         .groupby('subject_id')['start'].agg(['min', 'max']))
lo, hi = pd.Timestamp(BATCH_SESSION[0]), pd.Timestamp(BATCH_SESSION[1]) + pd.Timedelta(days=1)
start = dates['min'].dt.tz_localize(None) if dates['min'].dt.tz is not None else dates['min']
dates['batch'] = np.where((start >= lo) & (start < hi), 'January 2013', 'other')
print(dates.sort_values('min').to_string())
JANUARY = set(dates.index[dates.batch == 'January 2013'])
OTHER = set(hc_sample) - JANUARY
print(f'\nHC in the January 2013 session: {len(JANUARY)} | outside: {len(OTHER)} {sorted(OTHER)}')

In [ ]:
by_feature = []
for cond in CONDITIONS:
    d = long_data[long_data.condition == cond].set_index('subject')
    hc_jan, hc_other = d.loc[sorted(JANUARY), FEATURES], d.loc[sorted(OTHER), FEATURES]
    mdd = d.loc[d.group == 'MDD', FEATURES]
    sig = layer1.loc[layer1[f'detected_{cond}'], ['feature', f'cliffs_delta_{cond}']]
    for f, d_group in sig.itertuples(index=False):
        by_feature.append({'condition': cond, 'feature': f, 'family': family_of(f),
                           'delta_group': d_group,
                           'delta_mdd_vs_hc_other': cliffs_delta(hc_other[f], mdd[f]),
                           'delta_batch': cliffs_delta(hc_other[f], hc_jan[f])})
by_feature = pd.DataFrame(by_feature)
by_feature.to_csv(os.path.join(RESULTS_DIR, 'batch_effect_by_feature.csv'), index=False)

by_family = (by_feature.groupby(['condition', 'family'])
             .agg(n=('feature', 'size'), delta_group=('delta_group', 'median'),
                  delta_mdd_vs_hc_other=('delta_mdd_vs_hc_other', 'median'), delta_batch=('delta_batch', 'median'))
             .reset_index())
by_family.to_csv(os.path.join(RESULTS_DIR, 'batch_effect_by_family.csv'), index=False)

for cond in CONDITIONS:
    x = by_feature[by_feature.condition == cond]
    same_sign = (np.sign(x.delta_mdd_vs_hc_other) == np.sign(x.delta_group)).mean()
    print(f'{cond}: {len(x)} significant features | rho(batch profile, group profile) = '
          f'{spearmanr(x.delta_batch, -x.delta_group)[0]:.2f} | same sign with HC outside the batch: {same_sign:.0%} | '
          f'median |delta|: all HC {x.delta_group.abs().median():.2f}, HC outside {x.delta_mdd_vs_hc_other.abs().median():.2f}')
    print(by_family[(by_family.condition == cond) & (by_family.n >= 5)]
          .sort_values('n', ascending=False).drop(columns='condition').round(2).to_string(index=False))